In [ ]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score
from xgboost import XGBClassifier
from imblearn.combine import SMOTEENN


In [ ]:
# Load dataset
data_path = r"C:\Users\mukhe\OneDrive\Desktop\Bank Fraud Detection\Data\bs140513_032310.csv"
df = pd.read_csv(data_path)

# Drop uninformative columns
df = df.drop(columns=["zipcodeOri", "zipMerchant", "merchant"])

In [ ]:
# Encode categorical features
label_encoders = {}
categorical_cols = ["customer", "age", "gender", "category"]
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

In [ ]:
# Enhanced Feature Engineering (15 features total)
df['log_amount'] = np.log(df['amount'] + 1)
df['sqrt_amount'] = np.sqrt(df['amount'])
df['amount_squared'] = df['amount'] ** 2
df['step_log'] = np.log(df['step'] + 1)

# Transaction size flags
df['is_small_transaction'] = (df['amount'] < 100).astype(int)
df['is_large_transaction'] = (df['amount'] > 2000).astype(int)
df['is_very_large_transaction'] = (df['amount'] > 10000).astype(int)

# Time-based flags
df['is_early_step'] = (df['step'] < 200).astype(int)
df['is_late_step'] = (df['step'] > 700).astype(int)

# Features and target
X = df.drop("fraud", axis=1)
y = df["fraud"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Apply SMOTE + ENN
from imblearn.combine import SMOTEENN
smote_enn = SMOTEENN(random_state=42)
X_train_res, y_train_res = smote_enn.fit_resample(X_train, y_train)

In [ ]:
# Enhanced XGBoost model with better hyperparameters
model = XGBClassifier(
    scale_pos_weight=(len(y_train_res) - sum(y_train_res)) / sum(y_train_res),
    max_depth=8,
    learning_rate=0.05,
    n_estimators=500,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    min_child_weight=5,
    gamma=0.1,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

# Fit with validation set for early stopping
X_val, X_train_final, y_val, y_train_final = train_test_split(
    X_train_res, y_train_res, test_size=0.8, random_state=42, stratify=y_train_res
)

model.fit(
    X_train_final, y_train_final,
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=50,
    verbose=50
)

In [ ]:
# Enhanced threshold optimization with multiple metrics
from sklearn.metrics import precision_score, recall_score, roc_auc_score

y_proba = model.predict_proba(X_test)[:, 1]
best_f1 = 0
best_thresh = 0.5
best_metrics = {}

print("Threshold Optimization Results:")
print("Threshold | Precision | Recall | F1-Score | ROC-AUC")
print("-" * 50)

for t in np.arange(0.3, 0.9, 0.025):
    y_pred = (y_proba >= t).astype(int)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    
    print(f"{t:.3f}     | {precision:.4f}    | {recall:.4f} | {f1:.4f}   | {roc_auc:.4f}")
    
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
        best_metrics = {
            'threshold': t,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'roc_auc': roc_auc
        }

In [ ]:
# Final evaluation with enhanced metrics
y_pred_final = (y_proba >= best_thresh).astype(int)

print(f"\n{'='*60}")
print("ENHANCED MODEL PERFORMANCE RESULTS")
print(f"{'='*60}")
print(f"Best Threshold: {best_thresh:.3f}")
print(f"Best F1-Score: {best_f1:.4f}")
print(f"\nOptimal Metrics:")
for key, value in best_metrics.items():
    print(f"{key.replace('_', ' ').title()}: {value:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred_final))

# Feature importance analysis
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

In [ ]:
# Save enhanced model and metadata
import json
from datetime import datetime

model_dir = "model"
os.makedirs(model_dir, exist_ok=True)

# Save model components
joblib.dump(model, os.path.join(model_dir, "fraud_model.pkl"))
joblib.dump(label_encoders, os.path.join(model_dir, "label_encoders.pkl"))
joblib.dump(best_thresh, os.path.join(model_dir, "threshold.pkl"))
joblib.dump(X.columns.tolist(), os.path.join(model_dir, "feature_names.pkl"))

# Save comprehensive training metadata
metadata = {
    "training_date": datetime.now().isoformat(),
    "model_type": "XGBoost Enhanced",
    "feature_count": len(X.columns),
    "training_samples": len(X_train_res),
    "test_samples": len(X_test),
    "best_threshold": best_thresh,
    "performance": {
        "accuracy": (y_pred_final == y_test).mean(),
        "precision": best_metrics['precision'],
        "recall": best_metrics['recall'],
        "f1_score": best_metrics['f1_score'],
        "roc_auc": best_metrics['roc_auc']
    },
    "feature_importance": [
        {"feature": row['feature'], "importance": row['importance']}
        for _, row in feature_importance.iterrows()
    ]
}

with open(os.path.join(model_dir, "training_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nModel saved successfully to {model_dir}/")
print(f"Features: {len(X.columns)} | Samples: {len(X_train_res):,}")
print(f"Performance: F1={best_f1:.4f}, ROC-AUC={best_metrics['roc_auc']:.4f}")

In [ ]:
print("Training columns:", X.columns.tolist())


Training columns: ['step', 'customer', 'age', 'gender', 'category', 'amount', 'log_amount', 'is_large_transaction']
